# Домашнее задание: очередь подозрительных окон аутентификации для SOC

**Источники и использование ИИ.** Генератор данных взят из приложения к заданию без изменений. Код конвейера я подготовил с помощью ИИ-ассистента (Perplexity), опираясь на документацию scikit-learn: `Pipeline`, `ColumnTransformer`, `OneHotEncoder(handle_unknown="ignore")`, `IsolationForest.score_samples`. Выводы и рекомендацию я сверил с результатами своего запуска. Я перезапустил ядро и выполнил notebook сверху вниз. Проверил результаты утверждений об отсутствии пересечений row_id, доступности меток и соблюдении суточного бюджета. Для Random Forest на valid вручную пересчитал precision@8 = 33/40 и recall@8 = 33/106. Направление оценки Isolation Forest проверил по документации.

## 0. Окружение
Фактические версии выводятся ячейкой ниже (проверенное окружение из задания — в разделе 9).

In [15]:
import sys, json, datetime, warnings
import numpy as np, pandas as pd, scipy, sklearn, joblib, threadpoolctl
warnings.filterwarnings("ignore")
def display(x):
    print(x.to_string() if hasattr(x, "to_string") else x)
print("Python", sys.version.split()[0])
for m in (np, pd, scipy, sklearn, joblib, threadpoolctl):
    print(f"{m.__name__}=={m.__version__}")
SEED = 42

Python 3.12.10
numpy==2.3.5
pandas==2.2.3
scipy==1.16.3
sklearn==1.7.2
joblib==1.5.2
threadpoolctl==3.6.0


## 1. Постановка задачи
Результатом пользуется аналитик SOC. Он вручную разбирает карточки окон и успевает не больше восьми за сутки UTC. Одна строка набора — одна учётная запись из двадцати в одном закрытом 15-минутном окне, и к суткам она относится по `window_end`. Положительный класс — `target = 1`, то есть окно, которое по сценарию генератора требует расследования. Блокировки и другие действия в задание не входят.

Бюджет общий: восемь окон на все двадцать записей и обе группы вместе. Если нужное окно не попадёт в очередь (FN), оно останется без расследования. Если аналитик проверит обычное окно (FP), он потратит впустую один из восьми слотов, и из-за этого нужное окно может в очередь не попасть.

Ранжирование подходит этой постановке, потому что размер очереди задан бюджетом, а не моделью. Порог на оценке не гарантирует объём (на valid я это покажу), а ежедневный top-8 гарантирует. Оценка модели задаёт только порядок разбора. Решение о том, есть ли угроза и что делать, остаётся за аналитиком.

## 2. Паспорт данных и аудит
**Паспорт.** Данные порождает `generate_auth_data(stage="homework")` из приложения (`numpy.random.default_rng(42)`). Период — с 1 по 26 января 2026 года UTC. В наборе 20 учётных записей: 16 офисных и 4 сервисные. Метка `target` указывает, что окно требует расследования по сценарию генератора, и становится доступной через 24 часа после закрытия окна (`label_ready_at = window_end + 1 сутки`). Признаки: `failed_logins`, `new_device`, `mean_gap_s` (около 1% случайных пропусков), `geo_change`, `night_share`, `peer_group`. В признаки не входят `row_id` — идентификатор для аудита и разрешения равных оценок; `window_end` — время для разбиения и суточного бюджета; `user_id` — идентификатор записи для аудита; `target` — целевая метка; `label_ready_at` — время доступности вердикта. Наличие метки в выгрузке не означает её доступность при принятии решения.

Ограничения: источник синтетический, а одни и те же записи встречаются во всех периодах. Повторить результат можно при тех же версиях библиотек и фиксированных seed (раздел 9).


In [16]:
# Источник: учебный генератор из приложения к заданию (без изменений).
FEATURES = ["failed_logins","new_device","mean_gap_s","geo_change","night_share","peer_group"]

def generate_auth_data(stage="homework"):
    """Synthetic 15-minute authentication windows, seed 42."""
    if stage not in {"homework", "session"}:
        raise ValueError("stage must be homework or session")
    rng = np.random.default_rng(42)
    times = pd.date_range("2026-01-01", periods=30 * 96, freq="15min", tz="UTC")
    window_end = times.repeat(20)
    user_no = np.tile(np.arange(20), len(times))
    service = user_no >= 16
    n = len(window_end)
    day_no = (window_end - times[0]).days.to_numpy() + 1
    threat_rate = np.where(day_no >= 27, 0.015, 0.012)
    target = rng.binomial(1, threat_rate, n)
    legitimate_batch = (rng.random(n) < 0.006) & (target == 0)
    failed = rng.poisson(np.where(service, 2.2, 1.2) + 2 * target)
    failed += legitimate_batch * rng.integers(5, 10, n)
    new_device = rng.binomial(1, np.where(target == 1, 0.45, np.where(service, 0.01, 0.08)))
    mean_gap = rng.gamma(2.0, 12.0, n)
    mean_gap[target == 1] = rng.gamma(2.0, 4.5, int(target.sum()))
    mean_gap[legitimate_batch] = rng.gamma(2.0, 1.0, int(legitimate_batch.sum()))
    geo_change = rng.binomial(1, np.where(target == 1, 0.25, np.where(service, 0.005, 0.02)))
    night_share = rng.beta(1.0, 5.0, n)
    night_share[service] = rng.beta(3.0, 3.0, int(service.sum()))
    night_share[target == 1] = rng.beta(2.5, 3.5, int(target.sum()))
    late_service = (day_no >= 27) & service & (target == 0)
    failed[late_service] += rng.binomial(1, 0.5, int(late_service.sum()))
    mean_gap[rng.random(n) < 0.01] = np.nan
    data = pd.DataFrame({
        "row_id": [f"AUTH-{i:06d}" for i in range(n)],
        "window_end": window_end,
        "user_id": [f"u{i:02d}" for i in user_no],
        "peer_group": np.where(service, "service", "office"),
        "failed_logins": failed, "new_device": new_device, "mean_gap_s": mean_gap,
        "geo_change": geo_change, "night_share": night_share, "target": target})
    data["label_ready_at"] = data["window_end"] + pd.Timedelta(days=1)
    if stage == "homework":
        data = data[data["window_end"] < pd.Timestamp("2026-01-27", tz="UTC")]
    return data.reset_index(drop=True)

df = generate_auth_data(stage="homework")
print(df.shape, df.window_end.min(), df.window_end.max())

(49920, 11) 2026-01-01 00:00:00+00:00 2026-01-26 23:45:00+00:00


## 3. Разбиение и протокол
Я режу данные по `window_end` в UTC: левая граница входит, правая нет. Окна из пауз 15 и 21 января не использую. Метки test я отделяю от признаков сразу после разбиения и не читаю до шага 7. Ниже проверки структуры, уникальности идентификаторов и доступности меток.

In [17]:
UTC = lambda s: pd.Timestamp(s, tz="UTC")
BOUNDS = {"train": (UTC("2026-01-01"), UTC("2026-01-15")),
          "valid": (UTC("2026-01-16"), UTC("2026-01-21")),
          "test":  (UTC("2026-01-22"), UTC("2026-01-27"))}
PAUSES = [(UTC("2026-01-15"), UTC("2026-01-16")), (UTC("2026-01-21"), UTC("2026-01-22"))]
NUM = ["failed_logins", "new_device", "mean_gap_s", "geo_change", "night_share"]
CAT = ["peer_group"]
FEATS = NUM + CAT
assert set(FEATS) == set(FEATURES)

def take(name):
    a, b = BOUNDS[name]
    return df[(df.window_end >= a) & (df.window_end < b)].copy()

train, valid, test_full = take("train"), take("valid"), take("test")
# Метки test изолируются сразу; до шага 7 не используются.
y_test_sealed = test_full["target"].copy()
test = test_full.drop(columns="target")

rows = []
for n, d in (("train", train), ("valid", valid), ("test", test)):
    r = dict(part=n, rows=len(d), first_window_end=d.window_end.min(), last_window_end=d.window_end.max())
    if n != "test":
        r.update(positives=int(d.target.sum()), missing_values=int(d[FEATS].isna().sum().sum()),
                 missing_mean_gap_s=int(d.mean_gap_s.isna().sum()))
    rows.append(r)
split_summary = pd.DataFrame(rows)
display(split_summary)

    part   rows          first_window_end           last_window_end  positives  missing_values  missing_mean_gap_s
0  train  26880 2026-01-01 00:00:00+00:00 2026-01-14 23:45:00+00:00      347.0           299.0               299.0
1  valid   9600 2026-01-16 00:00:00+00:00 2026-01-20 23:45:00+00:00      106.0            98.0                98.0
2   test   9600 2026-01-22 00:00:00+00:00 2026-01-26 23:45:00+00:00        NaN             NaN                 NaN


In [18]:
audit = []
def check(name, ok, detail=""):
    audit.append({"check": name, "passed": bool(ok), "detail": detail})

for n, d in (("train", train), ("valid", valid)):
    check(f"{n}: row_id уникален", d.row_id.is_unique, f"{d.row_id.nunique()}/{len(d)}")
    check(f"{n}: target только 0/1", set(d.target.unique()) <= {0, 1})
    check(f"{n}: new_device, geo_change бинарные", d[["new_device","geo_change"]].isin([0,1]).all().all())
    check(f"{n}: night_share в [0,1]", d.night_share.between(0, 1).all())
    check(f"{n}: failed_logins целое и >= 0", pd.api.types.is_integer_dtype(d.failed_logins) and (d.failed_logins >= 0).all(),
          f"max={d.failed_logins.max()}")
    check(f"{n}: mean_gap_s > 0 (без учёта NaN)", (d.mean_gap_s.dropna() > 0).all(),
          f"min={d.mean_gap_s.min():.3f}, NaN={d.mean_gap_s.isna().sum()}")
    check(f"{n}: пропуски только в mean_gap_s", d[FEATS].isna().sum().drop("mean_gap_s").sum() == 0)
    check(f"{n}: peer_group in {{office, service}}", set(d.peer_group.unique()) == {"office", "service"},
          str(d.peer_group.value_counts().to_dict()))
    check(f"{n}: window_end в границах", d.window_end.min() >= BOUNDS[n][0] and d.window_end.max() < BOUNDS[n][1])
    check(f"{n}: window_end tz-aware UTC", str(d.window_end.dt.tz) == "UTC")
    check(f"{n}: доля положительных", True, f"{d.target.mean():.4f} ({int(d.target.sum())}/{len(d)})")
    check(f"{n}: нет пустых суток", d.window_end.dt.floor("D").nunique() == (14 if n == "train" else 5), f"суток={d.window_end.dt.floor('D').nunique()}")
check("test: нужные столбцы без target", set(FEATS + ["row_id", "window_end", "label_ready_at"]) <= set(test.columns) and "target" not in test.columns)
check("test: row_id уникален", test.row_id.is_unique)
check("test: window_end в границах", test.window_end.min() >= BOUNDS["test"][0] and test.window_end.max() < BOUNDS["test"][1])
check("всего 20 учётных записей", df.user_id.nunique() == 20)
check("в наборе нет окон после 26.01 (stage=homework)", df.window_end.max() < UTC("2026-01-27"))
audit_df = pd.DataFrame(audit)
display(audit_df)
assert audit_df.passed.all()
print("Состав групп по user:", train.groupby("peer_group").user_id.nunique().to_dict())
display(pd.concat({"train": train.groupby("peer_group").target.agg(["size","sum","mean"]),
                   "valid": valid.groupby("peer_group").target.agg(["size","sum","mean"])}))

                                             check  passed                              detail
0                           train: row_id уникален    True                         26880/26880
1                         train: target только 0/1    True                                    
2           train: new_device, geo_change бинарные    True                                    
3                       train: night_share в [0,1]    True                                    
4                train: failed_logins целое и >= 0    True                              max=14
5            train: mean_gap_s > 0 (без учёта NaN)    True                  min=0.042, NaN=299
6              train: пропуски только в mean_gap_s    True                                    
7           train: peer_group in {office, service}    True  {'office': 21504, 'service': 5376}
8                     train: window_end в границах    True                                    
9                   train: window_end tz-aware UTC

In [19]:
sets = [set(train.row_id), set(valid.row_id), set(test.row_id)]
assert not (sets[0] & sets[1]) and not (sets[0] & sets[2]) and not (sets[1] & sets[2]), "пересечение row_id"
assert train.label_ready_at.max() <= BOUNDS["valid"][0], "метки train недоступны к началу valid"
assert valid.label_ready_at.max() <= BOUNDS["test"][0], "метки valid недоступны к началу test"
for a, b in PAUSES:
    for d in (train, valid, test):
        assert not ((d.window_end >= a) & (d.window_end < b)).any(), "строки паузы использованы"
assert ((df.label_ready_at - df.window_end) == pd.Timedelta("24h")).all()
print("max label_ready_at train:", train.label_ready_at.max(), "<= start valid", BOUNDS["valid"][0])
print("max label_ready_at valid:", valid.label_ready_at.max(), "<= start test ", BOUNDS["test"][0])
print("Все проверки протокола пройдены.")

max label_ready_at train: 2026-01-15 23:45:00+00:00 <= start valid 2026-01-16 00:00:00+00:00
max label_ready_at valid: 2026-01-21 23:45:00+00:00 <= start test  2026-01-22 00:00:00+00:00
Все проверки протокола пройдены.


**Результат.** В train 26 880 строк (347 положительных), в valid 9 600 (106), в test 9 600. Все проверки аудита и протокола прошли. Пересечений `row_id` нет, метки train готовы к 15.01 23:45 (до начала valid), метки valid — к 21.01 23:45 (до начала test). Пропуски есть только в `mean_gap_s`. Сводку test с метками я дополняю после фиксации решения (шаг 7).

## 4. Подготовка и классификаторы
Подготовку числовых и категориальных полей я объединил с моделью в `Pipeline` и `ColumnTransformer`. Пропуски в числовых признаках заполняются медианой по train. Для Logistic Regression и Isolation Forest я дополнительно применяю `StandardScaler` (параметры тоже по train). Линейная модель чувствительна к масштабу признаков и регуляризации. Пропуски я заполняю медианой, рассчитанной на train. Для Logistic Regression использую StandardScaler, поскольку масштаб признаков влияет на регуляризацию и численное обучение модели. Для Random Forest масштабирование не применяю. В Pipeline Isolation Forest масштабирование оставлено как часть выбранной подготовки, однако для его случайных разбиений оно обычно не является необходимым. Параметры всех преобразований оцениваются только на train. Для Random Forest масштабирование избыточно: дерево не меняется при монотонном преобразовании отдельного признака. Категорию `peer_group` кодирует `OneHotEncoder(handle_unknown="ignore")`. Новое значение при применении даёт нулевой вектор, и я проверил это на значении `contractor`.

**Logistic Regression** — baseline: `C=1.0`, `max_iter=1000`, `random_state=42`. `class_weight` я не задавал, чтобы вероятности можно было сравнивать с порогами 0,1, 0,3 и 0,5. **Random Forest**: `n_estimators=200`, `random_state=42`. Я проверил шесть вариантов: `min_samples_leaf` из {5, 20, 50} и `max_depth` из {8, None}. Выбирал по train → valid, критерий — число TP в ежедневном top-8 на valid. Test для настройки не использовал.

In [20]:
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, IsolationForest

def make_prep(scale):
    num_steps = [("impute", SimpleImputer(strategy="median"))]
    if scale:
        num_steps.append(("scale", StandardScaler()))
    return ColumnTransformer([("num", Pipeline(num_steps), NUM),
                              ("cat", OneHotEncoder(handle_unknown="ignore"), CAT)])

X_train, y_train = train[FEATS], train.target
X_valid, y_valid = valid[FEATS], valid.target

LR_PARAMS = dict(C=1.0, max_iter=1000, random_state=SEED)
logreg = Pipeline([("prep", make_prep(True)), ("model", LogisticRegression(**LR_PARAMS))]).fit(X_train, y_train)
score_lr = logreg.predict_proba(X_valid)[:, 1]

probe = X_valid.head(3).copy(); probe["peer_group"] = "contractor"
assert logreg.predict_proba(probe).shape == (3, 2)
print("Новая категория peer_group обработана без ошибки.")

Новая категория peer_group обработана без ошибки.


In [21]:
BUDGET = 8
RULE = "daily top-8 UTC; score desc, row_id asc"

def daily_queue(d, score, k=BUDGET):
    q = d[["row_id", "window_end"]].copy()
    q["score"] = np.asarray(score)
    q["day"] = q.window_end.dt.floor("D")
    q = q.sort_values(["day", "score", "row_id"], ascending=[True, False, True])
    return q.groupby("day", group_keys=False).head(k)

def evaluate(d, q, label, rule=RULE):
    y = d.set_index("row_id").target
    tp = int(y.loc[q.row_id].sum()); n = len(q); P = int(y.sum())
    return {"method": label, "period": f"{d.window_end.min().date()} .. {d.window_end.max().date()}",
            "windows": len(d), "positives": P, "queue_size": n,
            "max_daily_load": int(q.groupby("day").size().max()), "TP": tp, "FP": n - tp,
            "precision@8": tp / n if n else "не определено: пустая очередь",
            "recall@8": tp / P if P else "не определено: нет положительных",
            "rule": rule}

def check_budget(d, q):
    cnt = q.groupby("day").size()
    exp = d.groupby(d.window_end.dt.floor("D")).size().clip(upper=BUDGET)
    assert (cnt == exp).all() and cnt.max() <= BUDGET
    return cnt

_t = pd.DataFrame({"row_id": ["B","A","C"], "window_end": pd.to_datetime(["2026-01-16 01:00"]*3, utc=True)})
assert list(daily_queue(_t, [1,1,1], k=2).row_id) == ["A","B"]   # равные оценки -> меньший row_id

In [22]:
RF_GRID = [dict(min_samples_leaf=m, max_depth=d) for m in (5, 20, 50) for d in (8, None)]
RF_BASE = dict(n_estimators=200, random_state=SEED, n_jobs=-1)
rf_rows = []
for p in RF_GRID:
    pipe = Pipeline([("prep", make_prep(False)), ("model", RandomForestClassifier(**RF_BASE, **p))]).fit(X_train, y_train)
    m = evaluate(valid, daily_queue(valid, pipe.predict_proba(X_valid)[:, 1]), "RF")
    rf_rows.append({**p, "TP": m["TP"], "precision@8": m["precision@8"], "recall@8": m["recall@8"]})
rf_tuning = pd.DataFrame(rf_rows)
display(rf_tuning)
best_i = int(rf_tuning.sort_values(["TP", "min_samples_leaf"], ascending=[False, False]).index[0])
RF_BEST = RF_GRID[best_i]
rf = Pipeline([("prep", make_prep(False)), ("model", RandomForestClassifier(**RF_BASE, **RF_BEST))]).fit(X_train, y_train)
score_rf = rf.predict_proba(X_valid)[:, 1]
print("Выбранные параметры RF:", RF_BEST)

   min_samples_leaf  max_depth  TP  precision@8  recall@8
0                 5        8.0  31        0.775  0.292453
1                 5        NaN  31        0.775  0.292453
2                20        8.0  32        0.800  0.301887
3                20        NaN  31        0.775  0.292453
4                50        8.0  31        0.775  0.292453
5                50        NaN  33        0.825  0.311321
Выбранные параметры RF: {'min_samples_leaf': 50, 'max_depth': None}


**Результат.** Варианты Random Forest дают 31–33 TP из 40, то есть разница небольшая. Я взял `min_samples_leaf=50`, `max_depth=None` (33 TP). Так как выбор сделан по valid, оценка на valid для него слегка оптимистична. Это стоит учитывать при сравнении с test. Разброс между вариантами всего 31–33 TP из 40, то есть 1–2 окна на пять суток. Такая разница вполне может быть следствием шума. 

## 5. Isolation Forest
Детектор и его преобразования я обучил только на признаках train: `target` не передавал и строки по метке не отбирал. Согласно документации, `score_samples` возвращает оценку, у которой меньшее значение означает более аномальное окно. Поэтому для приоритета я беру `-score_samples`. Проверка ниже: у окон с `target=1` средний приоритет выше.

Аномалия здесь означает редкую комбинацию признаков, а не угрозу. В генераторе есть легитимные пакеты: обычные окна с большим числом неудачных входов и очень коротким `mean_gap_s`. Они редки, но безопасны, и детектор может ставить их высоко.

Здесь мы проверяем другой подход: Isolation Forest ищет не окна, похожие на известные угрозы, а окна, которые выделяются среди остальных. При обучении ему не говорят, где угроза, а где обычная активность. 

In [23]:
ISO_PARAMS = dict(n_estimators=300, contamination="auto", random_state=SEED)
iso = Pipeline([("prep", make_prep(True)), ("model", IsolationForest(**ISO_PARAMS))]).fit(X_train)  # без target
raw_if = iso.score_samples(X_valid)       # по документации: чем ниже, тем аномальнее
score_if = -raw_if                        # больше = выше приоритет
print("Среднее raw score_samples: target=1 %.4f | target=0 %.4f" % (raw_if[y_valid==1].mean(), raw_if[y_valid==0].mean()))
print("Среднее priority score:    target=1 %.4f | target=0 %.4f" % (score_if[y_valid==1].mean(), score_if[y_valid==0].mean()))

Среднее raw score_samples: target=1 -0.5695 | target=0 -0.4505
Среднее priority score:    target=1 0.5695 | target=0 0.4505


На valid средний приоритет положительных окон составил 0,5695, обычных — 0,4505. Таким образом, детектор в среднем присваивает более высокие оценки окнам, требующим расследования. Однако этого недостаточно для качественной очереди: в ежедневный top-8 попали только 13 положительных окон из 40 выбранных. Следовательно, при заданном бюджете оценка аномальности уступает оценке обученного классификатора.

## 6. Очереди и оценка на valid
Основное правило — ежедневный top-8 по UTC. Внутри суток окна сортируются по оценке по убыванию, равные оценки разрешаются по возрастанию `row_id`, в очередь берутся первые `min(8, число окон за сутки)`. Метрики считаются по всему valid после ежедневного отбора: precision@8 = TP / число выбранных окон, recall@8 = TP / все положительные окна valid (106). Метки присоединяются к очереди уже после её построения.

In [24]:
scores_valid = {"Logistic Regression": score_lr, "Random Forest": score_rf, "Isolation Forest": score_if}
queues_valid = {k: daily_queue(valid, v) for k, v in scores_valid.items()}
loads = pd.DataFrame({k: check_budget(valid, q) for k, q in queues_valid.items()})
display(loads)
valid_table = pd.DataFrame([evaluate(valid, queues_valid[k], k) for k in scores_valid])
display(valid_table.drop(columns="rule"))
print("Правило отбора во всех строках:", RULE)
print("Положительных по суткам valid:", valid.groupby(valid.window_end.dt.floor("D")).target.sum().to_dict())

                           Logistic Regression  Random Forest  Isolation Forest
day                                                                            
2026-01-16 00:00:00+00:00                    8              8                 8
2026-01-17 00:00:00+00:00                    8              8                 8
2026-01-18 00:00:00+00:00                    8              8                 8
2026-01-19 00:00:00+00:00                    8              8                 8
2026-01-20 00:00:00+00:00                    8              8                 8
                method                    period  windows  positives  queue_size  max_daily_load  TP  FP  precision@8  recall@8
0  Logistic Regression  2026-01-16 .. 2026-01-20     9600        106          40               8  23  17        0.575  0.216981
1        Random Forest  2026-01-16 .. 2026-01-20     9600        106          40               8  33   7        0.825  0.311321
2     Isolation Forest  2026-01-16 .. 2026-01-20     960

При одинаковом суточном бюджете Random Forest формирует наиболее полезную очередь на valid: 33 нужных окна из 40 выбранных. Однако ограниченный бюджет позволяет охватить только 31,1% всех положительных окон периода.

In [25]:
best_clf = max(["Logistic Regression", "Random Forest"], key=lambda k: int(valid_table.set_index("method").loc[k, "TP"]))
s_thr = scores_valid[best_clf]
day_v = valid.window_end.dt.floor("D")
rows = []
for t in (0.1, 0.3, 0.5):
    sel = valid[s_thr >= t]; n = len(sel); tp = int(sel.target.sum())
    per_day = sel.groupby(sel.window_end.dt.floor("D")).size().reindex(day_v.unique(), fill_value=0)
    rows.append({"classifier": best_clf, "threshold": t, "selected": n, "max_daily": int(per_day.max()),
                 "days_over_8": int((per_day > 8).sum()), "excess_over_budget": int((per_day - 8).clip(lower=0).sum()),
                 "TP": tp, "FP": n - tp,
                 "precision": tp / n if n else "не определено: ничего не выбрано",
                 "recall": tp / valid.target.sum(), "rule": "score >= threshold, без лимита"})
display(pd.DataFrame(rows))

      classifier  threshold  selected  max_daily  days_over_8  excess_over_budget  TP   FP                         precision    recall                            rule
0  Random Forest        0.1       290         64            5                 250  75  215                          0.258621  0.707547  score >= threshold, без лимита
1  Random Forest        0.3        13          6            0                   0  12    1                          0.923077  0.113208  score >= threshold, без лимита
2  Random Forest        0.5         0          0            0                   0   0    0  не определено: ничего не выбрано  0.000000  score >= threshold, без лимита


### Оценка Isolation Forest на valid и контрольная выборка (шаг 5)

In [26]:
q_if = queues_valid["Isolation Forest"].merge(valid[["row_id","target","failed_logins","mean_gap_s","peer_group"]], on="row_id")
fp = q_if[q_if.target == 0]
print("IF top-8: FP =", len(fp), "| из них failed_logins >= 5 и mean_gap_s < 6 (похоже на легитимный пакет):",
      int(((fp.failed_logins >= 5) & (fp.mean_gap_s < 6)).sum()))
display(q_if.groupby("peer_group").agg(selected=("row_id","size"), tp=("target","sum")))
outside = valid[~valid.row_id.isin(queues_valid["Isolation Forest"].row_id)].copy()
outside["day"] = outside.window_end.dt.floor("D")
ctrl = outside.groupby("day", group_keys=False).sample(n=20, random_state=SEED)  # 20 окон/сутки вне очереди
p = ctrl.target.mean(); se = np.sqrt(p * (1 - p) / len(ctrl))
print(f"Контрольная выборка вне очереди: {len(ctrl)} окон, положительных {int(ctrl.target.sum())}, доля {p:.3f} (±{1.96*se:.3f}, приближённо)")
print("Истинная доля положительных вне очереди (доступна только в учебном наборе): %.4f" % outside.target.mean())

IF top-8: FP = 27 | из них failed_logins >= 5 и mean_gap_s < 6 (похоже на легитимный пакет): 3
            selected  tp
peer_group              
office            26  11
service           14   2
Контрольная выборка вне очереди: 100 окон, положительных 1, доля 0.010 (±0.020, приближённо)
Истинная доля положительных вне очереди (доступна только в учебном наборе): 0.0097


**Результат и ограничения.** В очереди Isolation Forest 27 FP из 40. Только 3 из них совпадают с моим грубым признаком легитимного пакета (`failed_logins >= 5` и `mean_gap_s < 6`). Значит, низкую точность нельзя объяснить одними пакетами: часть редких окон просто не связана с целевым сценарием.

**План ручной проверки при неполной разметке.** Каждые сутки аналитик разбирает top-K (K=8) и дополнительно небольшую случайную выборку окон вне очереди (в учебной проверке — 20 окон в сутки, seed 42). Выборка вне очереди даёт грубую оценку доли положительных среди невыбранных окон: в 100 окнах нашлось одно положительное (1,0%, интервал примерно ±2 п.п.), а настоящая доля в valid составляет 0,97%. По такому плану можно сделать вывод о precision очереди и о доле пропусков в порядке величины. Неопределёнными остаются полнота по всему потоку (интервал слишком широк для редких событий), редкие сценарии, не попавшие в выборку, и качество вердиктов самих аналитиков. Невыбранные окна  не считаем автоматически обычными.

**Результат.** Нагрузка в каждые сутки ровно 8 окон, всего 40.

| Метод | TP | FP | precision@8 | recall@8 |
|---|---|---|---|---|
| Random Forest | 33 | 7 | 0,825 | 0,311 |
| Logistic Regression | 23 | 17 | 0,575 | 0,217 |
| Isolation Forest | 13 | 27 | 0,325 | 0,123 |

**Пороговый опыт** (Random Forest, правило «оценка ≥ порога», без лимита). При пороге 0,1 выбрано 290 окон: превышение бюджета 250 окон, все пять суток выше 8, максимум 64 в сутки, precision 0,259, recall 0,708. При 0,3 выбрано 13 окон (12 TP, бюджет не превышен), precision 0,923, recall 0,113. При 0,5 не выбрано ничего, поэтому precision не определена (деление на ноль), а recall равна 0. Порог управляет качеством, но не нагрузкой. Для итогового решения я использую top-8.

**Интерпретация.** В valid бывает от 17 до 26 положительных окон в сутки при бюджете 8, поэтому recall@8 физически ограничен сверху примерно 0,35–0,4. Значение 0,31 у Random Forest близко к этому потолку.

## 7. Зафиксированный выбор и итоговый test
Кандидата я выбираю по valid: максимум TP в top-8, а при равенстве — более простую модель. Конфигурацию (модель, параметры, признаки, версии библиотек, правило top-8) записываю в `frozen_config.json` до расчёта test. Затем применяю модель к test без дообучения и без изменения правила. Метки test присоединяю только в последней ячейке.

In [27]:
winner = valid_table.assign(_ord=[1, 2, 3]).sort_values(["TP", "_ord"], ascending=[False, True]).iloc[0].method
final_models = {"Logistic Regression": (logreg, LR_PARAMS), "Random Forest": (rf, {**RF_BASE, **RF_BEST}),
                "Isolation Forest": (iso, ISO_PARAMS)}
frozen = {"frozen_at_utc": datetime.datetime.now(datetime.timezone.utc).isoformat(timespec="seconds"),
          "model": winner, "params": final_models[winner][1], "features": FEATS, "rule": RULE,
          "selection_basis": "max TP@8 на valid; при равенстве - более простая модель",
          "valid_TP": {r.method: int(r.TP) for r in valid_table.itertuples()},
          "versions": {m.__name__: m.__version__ for m in (np, pd, scipy, sklearn, joblib, threadpoolctl)},
          "test_labels_used_before_freeze": False}
with open("frozen_config.json", "w") as f: json.dump(frozen, f, indent=2, default=str, ensure_ascii=False)
print(json.dumps(frozen, indent=2, default=str, ensure_ascii=False))

{
  "frozen_at_utc": "2026-11-23T17:55:19+00:00",
  "model": "Random Forest",
  "params": {
    "n_estimators": 200,
    "random_state": 42,
    "n_jobs": -1,
    "min_samples_leaf": 50,
    "max_depth": null
  },
  "features": [
    "failed_logins",
    "new_device",
    "mean_gap_s",
    "geo_change",
    "night_share",
    "peer_group"
  ],
  "rule": "daily top-8 UTC; score desc, row_id asc",
  "selection_basis": "max TP@8 на valid; при равенстве - более простая модель",
  "valid_TP": {
    "Logistic Regression": 23,
    "Random Forest": 33,
    "Isolation Forest": 13
  },
  "versions": {
    "numpy": "2.3.5",
    "pandas": "2.2.3",
    "scipy": "1.16.3",
    "sklearn": "1.7.2",
    "joblib": "1.5.2",
    "threadpoolctl": "3.6.0"
  },
  "test_labels_used_before_freeze": false
}


In [28]:
model = final_models[winner][0]
X_test = test[FEATS]
score_test = -model.score_samples(X_test) if winner == "Isolation Forest" else model.predict_proba(X_test)[:, 1]
q_test = daily_queue(test, score_test)
check_budget(test, q_test)
test_eval = test.assign(target=y_test_sealed)     # метки присоединяются только здесь
res = evaluate(test_eval, q_test, winner)
res["precision_target_0.50_met"] = res["precision@8"] >= 0.50
res["recall_target_0.10_met"] = res["recall@8"] >= 0.10
display(pd.DataFrame([res]))
test_summary = split_summary.copy()
test_summary.loc[test_summary.part == "test", ["positives", "missing_values", "missing_mean_gap_s"]] = [
    int(y_test_sealed.sum()), int(test[FEATS].isna().sum().sum()), int(test.mean_gap_s.isna().sum())]
display(test_summary)
print("Положительных по суткам test:", test_eval.groupby(test_eval.window_end.dt.floor("D")).target.sum().to_dict())

          method                    period  windows  positives  queue_size  max_daily_load  TP  FP  precision@8  recall@8                                     rule  precision_target_0.50_met  recall_target_0.10_met
0  Random Forest  2026-01-22 .. 2026-01-26     9600        106          40               8  32   8          0.8  0.301887  daily top-8 UTC; score desc, row_id asc                       True                    True
    part   rows          first_window_end           last_window_end  positives  missing_values  missing_mean_gap_s
0  train  26880 2026-01-01 00:00:00+00:00 2026-01-14 23:45:00+00:00      347.0           299.0               299.0
1  valid   9600 2026-01-16 00:00:00+00:00 2026-01-20 23:45:00+00:00      106.0            98.0                98.0
2   test   9600 2026-01-22 00:00:00+00:00 2026-01-26 23:45:00+00:00      106.0           109.0               109.0
Положительных по суткам test: {Timestamp('2026-01-22 00:00:00+0000', tz='UTC'): 12, Timestamp('2026-01-23 00:00:

**Результат на test** (правило: ежедневный top-8 UTC, оценка по убыванию, равенства по возрастанию `row_id`). Random Forest: 40 окон в очереди, максимум 8 в сутки, TP 32, FP 8, precision@8 0,800, recall@8 0,302 при 106 положительных. Учебные ориентиры (precision@8 ≥ 0,50 и recall@8 ≥ 0,10) достигнуты, но это не разрешение на внедрение. Результат близок к valid (0,825 и 0,311), заметной деградации я не вижу. Если понадобится новый эксперимент, его нужно проверять на новом периоде, а не подстраивать под этот test.

## 8. Рекомендация SOC
Необходимо развивать модель по стратегии с Random Forest и ежедневным top-8 в режиме приоритизации, без автоматических блокировок.

На valid Random Forest выбирает 33 подтверждённых окна из 40 (precision@8 0,825, recall@8 0,311). Logistic Regression находит 23, Isolation Forest — 13. На test результат устойчив: 32 из 40, precision@8 0,800, recall@8 0,302, нагрузка ровно 8 окон в сутки. Разница между вариантами Random Forest на valid мала, поэтому конкретные гиперпараметры я не считаю определяющими. Ансамбль при этом заметно лучше baseline по числу найденных окон.

Ошибки бывают двух видов. В test 8 из 40 карточек оказались обычными окнами (FP). Из 106 положительных окон очередь покрыла примерно 30%, остальные около 70% остались без расследования. Высокая доля полезных карточек при малом бюджете сочетается с низкой полнотой: в сутки 12–28 положительных окон при восьми слотах. Порог 0,3 даёт более чистую выборку (precision 0,923), но всего 13 окон за пять суток, а порог 0,1 выбирает 290 окон и сильно превышает бюджет. Поэтому порог не заменяет очередь. Isolation Forest уступает: «редкое» не значит «вредоносное», и 27 из 40 его карточек оказались обычными окнами.

В пилоте я бы использовал очередь только как порядок разбора внутри суточного бюджета, вёл журнал вердиктов аналитиков и ежедневно добавлял небольшую случайную проверку окон вне очереди. Так можно оценивать долю пропусков и вовремя заметить ухудшение качества.

Ограничения остаются существенными: данные синтетические, учётные записи те же, что в обучении, сдвига распределения в этом наборе нет, режим пакетный (очередь строится после окончания суток), а метки генератора надёжнее реальных вердиктов. До работы с реальной телеметрией нужно:
1. Проверить модель на новых учётных записях и на более позднем периоде со сдвигом поведения.
2. Оценить задержку и полноту вердиктов аналитиков и ввести случайную контрольную выборку вне очереди, а для потока отдельно определить правило обслуживания.

Контрольную выборку вне очереди следует проверять за счёт отдельно согласованного ресурса, не увеличивая основной бюджет незаметно для SOC. До получения результатов на реальной телеметрии не рекомендуется использовать модель для автоматических действий.


## 9. Окружение
Проверенное окружение из задания: Python 3.12, numpy 2.3.5, pandas 2.2.3, scipy 1.16.3, scikit-learn 1.7.2, joblib 1.5.2, threadpoolctl 3.6.0 (см. `requirements.txt`). Фактические версии моего запуска выведены в первой ячейке. 